[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/schelling_segregation_2026.ipynb)

# Schelling: individual choices, collective patterns
IPHS 200 · Thursday, October 29, 2026

**Can a preference for some similar neighbors produce a much more segregated neighborhood?** Predict the result before running the model.

Two groups live on a grid. An agent stays if a minimum fraction of its occupied neighboring cells belong to its own group; otherwise it moves to a random empty cell. Agents do not choose a new home based on its neighbors. We repeat these decisions and observe the larger pattern.

Adapted from Katherine Elkins’s 2025 Schelling notebook. Based on Thomas Schelling’s [Dynamic Models of Segregation (1971)](https://www.stat.berkeley.edu/~aldous/157/Papers/Schelling_Seg_Models.pdf) and the [Mesa Schelling example](https://mesa.readthedocs.io/stable/examples/basic/schelling.html).

## 1. Setup
Run once. No GPU is needed.

In [ ]:
%pip install -q "mesa==3.3.1" numpy pandas matplotlib

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display
from mesa import Agent, Model
from mesa.space import SingleGrid

print("Setup ready.")

## 2. Model rules
- Each cell holds one agent or is empty. The grid wraps at the edges.
- Neighbors are the eight surrounding cells; empty cells do not count toward the similarity fraction.
- Agents with no neighbors stay put. Otherwise they stay when the fraction of same-group neighbors meets the threshold.
- Each round activates all agents once in random order. Moves happen immediately; later agents see the updated grid.
- Satisfaction is checked again after the round. Stop when everyone is satisfied or the round limit is reached.

A threshold of **0.40** means at least 40% of occupied neighbors must match. With eight occupied neighbors, that requires four matches; **0.375** would require three. This is a fraction rule, not a fixed number of neighbors.

In [ ]:
class SchellingAgent(Agent):
    def __init__(self, model, group):
        super().__init__(model)
        self.group = int(group)

    def similar_fraction(self):
        neighbors = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        if not neighbors:
            return None
        return sum(other.group == self.group for other in neighbors) / len(neighbors)

    def is_satisfied(self):
        fraction = self.similar_fraction()
        return fraction is None or fraction >= self.model.threshold

    def step(self):
        if not self.is_satisfied():
            self.model.grid.move_to_empty(self)
            self.model.moves_this_round += 1


class SchellingModel(Model):
    def __init__(self, size=20, density=0.8, group_b_share=0.5, threshold=0.4, seed=200):
        if not isinstance(size, int) or not 5 <= size <= 60:
            raise ValueError("Grid size must be an integer from 5 to 60.")
        if not 0 < density < 1:
            raise ValueError("Density must be between 0 and 1, leaving at least one empty cell.")
        if not 0 < group_b_share < 1 or not 0 <= threshold <= 1:
            raise ValueError("Group B share must be between 0 and 1; threshold must be between 0 and 1 inclusive.")
        if not isinstance(seed, int):
            raise ValueError("Seed must be an integer.")
        population = round(size * size * density)
        group_b = round(population * group_b_share)
        if not 2 <= population < size * size or not 1 <= group_b < population:
            raise ValueError("Choose settings with both groups present and at least one empty cell.")
        super().__init__(seed=seed)
        self.size = size
        self.threshold = float(threshold)
        self.grid = SingleGrid(size, size, torus=True)
        self.population = population
        self.group_b_count = group_b
        self.round_number = 0
        self.moves_this_round = 0
        self.history = []
        positions = self.random.sample([(x, y) for x in range(size) for y in range(size)], population)
        groups = [0] * (population - group_b) + [1] * group_b
        self.random.shuffle(groups)
        for position, group in zip(positions, groups):
            self.grid.place_agent(SchellingAgent(self, group), position)
        self.initial_grid = self.grid_array()
        self.record_state()
        self.running = self.history[-1]["satisfied_pct"] < 100

    def grid_array(self):
        image = np.zeros((self.size, self.size), dtype=int)
        for agent in self.agents:
            x, y = agent.pos
            image[y, x] = agent.group + 1
        return image

    def record_state(self):
        fractions = [agent.similar_fraction() for agent in self.agents]
        connected = [fraction for fraction in fractions if fraction is not None]
        self.history.append({
            "round": self.round_number,
            "satisfied_pct": 100 * sum(agent.is_satisfied() for agent in self.agents) / self.population,
            "mean_same_group_pct": 100 * float(np.mean(connected)) if connected else np.nan,
            "all_same_neighbors_pct": 100 * sum(fraction == 1 for fraction in connected) / self.population,
            "isolated_agents": sum(fraction is None for fraction in fractions),
            "moves": self.moves_this_round,
        })

    def step(self):
        if not self.running:
            return
        self.moves_this_round = 0
        activation = list(self.agents)
        self.random.shuffle(activation)
        for agent in activation:
            agent.step()
        self.round_number += 1
        self.record_state()  # Recheck the final grid, not flags from earlier turns.
        self.running = self.history[-1]["satisfied_pct"] < 100


def run_model(size, density, group_b_share, threshold, seed, max_rounds):
    if not isinstance(max_rounds, int) or max_rounds < 1 or max_rounds > 1000:
        raise ValueError("Choose a round limit from 1 to 1000.")
    model = SchellingModel(size, density, group_b_share, threshold, seed)
    for _ in range(max_rounds):
        if not model.running:
            break
        model.step()
    return model

## 3. Run and compare
Change the settings below and rerun this cell. Each run starts a new model.

Keep the **seed** fixed while comparing thresholds so you start with the same people in the same places. Change it afterward to see whether the pattern persists in another random starting arrangement.

In [ ]:
#@title Model settings
Grid_Size = 20 #@param {type:"integer"}
Density = 0.8 #@param {type:"number"}
Group_B_Share = 0.5 #@param {type:"number"}
Similarity_Threshold = 0.4 #@param {type:"number"}
Seed = 200 #@param {type:"integer"}
Max_Rounds = 100 #@param {type:"integer"}

model = run_model(Grid_Size, Density, Group_B_Share, Similarity_Threshold, Seed, Max_Rounds)
results = pd.DataFrame(model.history)

cmap = ListedColormap(["#eeeeee", "#0072B2", "#D55E00"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, image, title in zip(axes[:2], [model.initial_grid, model.grid_array()],
                            ["Initial arrangement", f"After {model.round_number} rounds"]):
    ax.imshow(image, cmap=cmap, norm=norm, origin="lower", interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].legend(handles=[Patch(color=cmap(i), label=label) for i, label in enumerate(["Empty", "Group A", "Group B"])],
               loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3)
axes[2].plot(results["round"], results["satisfied_pct"], label="Satisfied agents (%)")
axes[2].plot(results["round"], results["mean_same_group_pct"], label="Mean same-group neighbors (%)")
axes[2].set(xlabel="Round", ylabel="Percent", ylim=(-2, 102), title="Local satisfaction and neighborhood composition")
axes[2].legend(loc="best", fontsize=8)
axes[2].grid(alpha=0.2)
fig.suptitle(f"Threshold {Similarity_Threshold:.3f} · density {Density:.2f} · Group B {model.group_b_count/model.population:.1%} · seed {Seed}")
plt.show()

status = "Everyone is satisfied." if not model.running else "Round limit reached; some agents remain unsatisfied."
print(f"{model.population} agents, {Grid_Size**2-model.population} empty cells. {status}")
display(results.iloc[[0, -1]].drop_duplicates(subset="round").round(2))
fig.savefig("schelling_comparison.png", dpi=160, bbox_inches="tight")
results.to_csv("schelling_history.csv", index=False)
Path("schelling_settings.json").write_text(json.dumps({
    "grid_size": Grid_Size, "density": Density, "group_b_share": Group_B_Share,
    "threshold": Similarity_Threshold, "seed": Seed, "max_rounds": Max_Rounds,
}, indent=2))
print("Saved the graph, history, and settings in the notebook’s Files pane. Rerunning replaces these files.")

### Read the results
**Satisfied** means an agent’s own threshold is met, not that the neighborhood is integrated.

**Mean same-group neighbors** averages each non-isolated agent’s fraction of same-group neighbors. **All-same neighbors** is the percentage of all agents whose occupied neighbors all match; isolated agents are excluded from that count. With unequal group sizes, substantial same-group contact can already occur by chance—compare with the initial arrangement.

## 4. Try three experiments
1. Keep the seed fixed. Compare thresholds **0.20, 0.40, and 0.60**. Predict each result first. Save the graphs you want to compare before rerunning.
2. Repeat one threshold with another seed. Does the outcome change?
3. Change either density or Group B’s share. Keep the other settings fixed.

What is surprising about the relation between individual preferences and the final pattern? Which rule or parameter helps explain it? What does the model leave out about real housing choices?

## 5. Optional: compare several starting arrangements
This runs the same four thresholds with five seeds each. The points show the mean final neighborhood similarity and the bars show the range across runs. A run that reaches its limit is not necessarily a stable outcome.

In [ ]:
#@title Optional repeated experiments
Run_Repeated_Experiments = False #@param {type:"boolean"}

if Run_Repeated_Experiments:
    trials = []
    for threshold in [0.2, 0.4, 0.6, 0.8]:
        for trial_seed in range(Seed, Seed + 5):
            trial = run_model(Grid_Size, Density, Group_B_Share, threshold, trial_seed, Max_Rounds)
            trials.append({"threshold": threshold, "seed": trial_seed,
                           "all_satisfied": not trial.running, **trial.history[-1]})
    trials = pd.DataFrame(trials)
    summary = trials.groupby("threshold")["mean_same_group_pct"].agg(["mean", "min", "max"])
    display(trials.round(2))
    fig_batch, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(summary.index, summary["mean"],
                yerr=[summary["mean"]-summary["min"], summary["max"]-summary["mean"]],
                fmt="o-", capsize=5)
    ax.set(xlabel="Similarity threshold", ylabel="Final mean same-group neighbors (%)",
           ylim=(0, 102), title="Five seeds per threshold: mean and observed range")
    ax.grid(alpha=0.2)
    plt.show()
    trials.to_csv("schelling_repeated_runs.csv", index=False)
else:
    print("Optional: enable Run_Repeated_Experiments to compare five seeds per threshold.")